# Google API config

# Compare_and_Update_site_roster

<a href="https://colab.research.google.com/github/PeaceAndLongLife/Analysis-Colab/blob/Development/notebooks/Compare_and_Update_site_roster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# @title ## Mount google drive and read in the SERVICE_ACCOUNT_FILE  {"form-width":"20%"}

# @markdown ---
# @markdown
# @markdown The `SERVICE_ACCOUNT_FILE` path is stored as a secret in google colab. If you do not have this sotred on your colab, contact the Admin: Travis Kregear at tkregear@pdx.edu

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# load SERVICE_ACCOUNT_FILE
from google.colab import userdata

SERVICE_ACCOUNT_FILE = userdata.get('SERVICE_ACCOUNT_FILE')

ModuleNotFoundError: No module named 'google'

In [ ]:
# @title ## Install pyDrive  {"form-width":"20%"}
# @markdown
# @markdown ---
# @markdown
# @markdown Thie is necessary if using the Google API to call files by their file_id
# !pip install PyDrive

In [ ]:
# @title ## GoogleDocumentManager class  {"form-width":"20%"}

# @markdown This Class is defined to connect to the Google API, read in specified csv files and write out amodified dataframes to a target folder.
# @markdown
# @markdown ---
# @markdown     Class Requirments:
# @markdown     - SERVICE_ACCOUNT_FILE
# @markdown
# @markdown
# @markdown ### Functions
# @markdown
# @markdown ---
# @markdown  #### `read_file`
# @markdown   This function reads in a csv from its document id and stores it internally as a dataframe.
# @markdown
# @markdown     Function Requirments:
# @markdown     1. document_id
# @markdown         The document ID is the uniquie identifier from the google drive file. It can be found in the url of the file.
# @markdown
# @markdown  #### `upload_dataframe_to_drive`
# @markdown     Function Requirements:
# @markdown     1. df_to_upload,
# @markdown         This defines the dataframe to be output to a csv file.
# @markdown
# @markdown     2. filename,
# @markdown         This defines the name of the csv file to be uploaded to the drive.
# @markdown
# @markdown     3. folder_id,
# @markdown         This defines the id of the parent directory the csv will be saved in on the drive.
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseUpload # Import MediaIoBaseUpload
import pandas as pd
import io

### Google Docs client manager
class GoogleDocumentManager():
    def __init__(self,
                 service_account_file=SERVICE_ACCOUNT_FILE,
                 document_id=None,
                 scopes=[
                    'https://www.googleapis.com/auth/drive'
                    ]
                 ):
        self.service_account_file = service_account_file
        self.document_id = document_id
        self.scopes = scopes

    # read Document
    def read_file(self, document_id, file_type='csv'):

        try:
            credentials = service_account.Credentials.from_service_account_file(
                    self.service_account_file,
                    scopes=self.scopes)

            # Build the Google DRIVE API service
            drive_service = build('drive', 'v3', credentials=credentials)
            print("Google Drive API service built successfully.")

            # Download the file content
            request = drive_service.files().get_media(fileId=document_id)
            file_content = request.execute()

            # Read file into DataFrame based on file_type
            if file_type.lower() == 'csv':
                df = pd.read_csv(io.BytesIO(file_content))
            elif file_type.lower() == 'xlsx':
                df = pd.read_excel(io.BytesIO(file_content))
            return df

        # Raise errors is necessary
        except HttpError as error:
            print(f"An HTTP error occurred: {error}")
            return None
        except FileNotFoundError:
            print(f"Error: Service account key file not found at '{self.service_account_file}'")
            print("Please ensure the path to your service account key is correct.")
            return None
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            return None

    def upload_dataframe_to_drive(self, df_to_upload, filename, folder_id):
        if folder_id == 'YOUR_GOOGLE_DRIVE_FOLDER_ID_HERE':
            print(f"\nWarning: Please replace 'YOUR_GOOGLE_DRIVE_FOLDER_ID_HERE' with your actual Google Drive Folder ID to save '{filename}'.")
            return None

        try:
            credentials = service_account.Credentials.from_service_account_file(
                    self.service_account_file,
                    scopes=self.scopes)

            drive_service = build('drive', 'v3', credentials=credentials)
            print(f"Google Drive API service built successfully for uploading '{filename}'.")

            csv_buffer = io.StringIO()
            df_to_upload.to_csv(csv_buffer, index=True)
            csv_content = csv_buffer.getvalue()

            # Wrap the CSV content in a MediaIoBaseUpload object
            media_body = MediaIoBaseUpload(io.BytesIO(csv_content.encode('utf-8')),
                                           mimetype='text/csv',
                                           resumable=True)

            file_metadata = {
                'name': filename,
                'parents': [folder_id],
                'mimeType': 'text/csv'
            }

            file = drive_service.files().create(
                body=file_metadata,
                media_body=media_body, # Use the wrapped media_body
                fields='id',
                supportsAllDrives=True,
            ).execute()

            print(f"Successfully uploaded '{filename}' to Google Drive (File ID: {file.get('id')}).")
            return file.get('id')

        except FileNotFoundError:
            print(f"Error: Service account key file not found at '{self.service_account_file}'")
            print("Please ensure the path to your service account key is correct.")
            return None
        except HttpError as error:
            print(f"An HTTP error occurred during upload: {error}")
            return None
        except Exception as e:
            print(f"An unexpected error occurred during upload: {e}")
            return None

In [ ]:
# @title ## Extract File ID {"form-width":"20%"}
# @markdown This Python function is designed to extract the file ID from a Google Drive shared link.

import re

def extract_file_id(shared_link):
    """
    Extracts the Google Drive file ID from a shared link.

    Args:
        shared_link (str): The Google Drive shared link.

    Returns:
        str: The extracted file ID, or None if not found.
    """
    # Pattern 1: https://drive.google.com/file/d/FILE_ID/view
    # Pattern 2: https://drive.google.com/open?id=FILE_ID
    # Pattern 3: https://docs.google.com/spreadsheets/d/FILE_ID/edit (and similar for other doc types)
    # Pattern 4: https://drive.google.com/drive/folders/FOLDER_ID
    patterns = [
        r"https:\/\/drive\.google\.com\/file\/d\/([a-zA-Z0-9_-]+)",
        r"https:\/\/drive\.google\.com\/open\?id=([a-zA-Z0-9_-]+)",
        r"https:\/\/docs\.google\.com\/(?:spreadsheets|document|presentation)\/d\/([a-zA-Z0-9_-]+)",
        r"https:\/\/drive\.google\.com\/drive\/folders\/([a-zA-Z0-9_-]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, shared_link)
        if match:
            return match.group(1)

    return None

print("The 'extract_file_id' function has been defined.")

# Read in Data

In [ ]:
# @title Import Canvas roster CSV and Banner Roster xlsx {"form-width":"20%"}
csv_link = "https://drive.google.com/file/d/17T5HCGWP4W8B3jKywOxZtkpltZ0GWgoh/view?usp=drive_link" # @param {"type":"string","placeholder":"Enter the link for the Canvas roster CSV file"}
xlsx_link = "https://docs.google.com/spreadsheets/d/19xTIMlMQK4gQRIJ5Ymj7pfxGelLIKng_/edit?usp=drive_link&ouid=114068857540291368901&rtpof=true&sd=true" # @param {"type":"string","placeholder":"Enter the link for the Banner roster XLSX file"}

# @markdown ---
# @markdown Link to download canvas roster:
# @markdown
# @markdown https://canvas.pdx.edu/courses/112556/groups#tab-17024

# @markdown ---
# @markdown Link tyo download Banner roster:
# @markdown
# @markdown https://app.banner.pdx.edu/StudentSelfService/ssb/classListApp/classListPage#!/202601/42564/courseDetails/classList/summaryView

csv_file_id = extract_file_id(csv_link)
xlsx_file_id = extract_file_id(xlsx_link)

print(f"Extracted CSV File ID: {csv_file_id}")
print(f"Extracted XLSX File ID: {xlsx_file_id}")

document_manager = GoogleDocumentManager()

print("\nAttempting to read CSV file...")
df_csv = document_manager.read_file(csv_file_id, file_type='csv')
if df_csv is not None:
    print("CSV DataFrame head:")
    # display(df_csv.head())
else:
    print("Failed to read CSV file.")

print("\nAttempting to read XLSX file...")
df_xlsx = document_manager.read_file(xlsx_file_id, file_type='xlsx')
if df_xlsx is not None:
    # Set the 14th row (index 13) as column headers
    new_header = df_xlsx.iloc[13]
    df_xlsx = df_xlsx[14:] # Remove rows up to and including the new header row
    df_xlsx.columns = new_header # Set the new header
    # Reset index after dropping rows
    df_xlsx = df_xlsx.reset_index(drop=True)
    print("XLSX DataFrame head after processing:")
    display(df_xlsx.head())
else:
    print("Failed to read XLSX file.")

# Process data for site formatting.

In [ ]:
# @title Combine dataframes {"form-width":"20%"}
# @markdown Combine the dataframes using the `login_id` field from the canvas CSV file and the `Email` from the banner .xlsx file.

import pandas as pd

# Ensure login_id and Email are strings and lowercased for consistent comparison
df_csv['login_id_clean'] = df_csv['login_id'].astype(str).str.strip().str.lower()
df_xlsx['Email_clean'] = df_xlsx['Email'].astype(str).str.strip().str.lower()

# Prepare an empty list to store *only* successfully merged rows
merged_data = []

# Keep track of matched indices to correctly identify unmatched rows later
matched_xlsx_indices = set()
matched_csv_indices = set()

# Iterate through each row of the CSV DataFrame
for idx_csv, row_csv in df_csv.iterrows():
    login_id = row_csv['login_id_clean']

    # Find potential matches in the XLSX DataFrame
    # Criteria: login_id is a substring of Email AND the xlsx row has not been matched yet
    potential_matches_xlsx = df_xlsx[
        df_xlsx['Email_clean'].apply(lambda email: login_id in email) &
        (~df_xlsx.index.isin(matched_xlsx_indices))
    ]

    if not potential_matches_xlsx.empty:
        # If multiple matches, we'll take the first one found.
        matched_row_xlsx = potential_matches_xlsx.iloc[0]

        # Combine the CSV row and the matched XLSX row
        combined_series = pd.concat([row_csv, matched_row_xlsx], ignore_index=False)
        merged_data.append(combined_series)

        # Add the index of the matched XLSX row to the set of matched indices
        matched_xlsx_indices.add(matched_row_xlsx.name)
        matched_csv_indices.add(idx_csv) # Mark CSV row as matched
    # IMPORTANT CHANGE: Do NOT add unmatched CSV rows to merged_data here.

# Create the initial merged DataFrame from *only* the matched rows
merged_df = pd.DataFrame(merged_data)

# Clean up the temporary columns used for matching
merged_df = merged_df.drop(columns=['login_id_clean', 'Email_clean'], errors='ignore')

print("Combined DataFrame head (only matched rows):")
# display(merged_df.head())
print("\nNumber of rows in combined DataFrame:", len(merged_df))

# Report unmatched rows (these were never added to merged_df)
print("\n--- Unmatched Rows ---")

unmatched_csv_df = df_csv[~df_csv.index.isin(matched_csv_indices)]
if not unmatched_csv_df.empty:
    print("\nRows from Canvas Roster (CSV) without a match in Banner Roster (XLSX):")
    # display(unmatched_csv_df)
else:
    print("\nAll Canvas Roster (CSV) rows found a match.")

unmatched_xlsx_df = df_xlsx[~df_xlsx.index.isin(matched_xlsx_indices)]
if not unmatched_xlsx_df.empty:
    print("\nRows from Banner Roster (XLSX) without a match in Canvas Roster (CSV):")
    display(unmatched_xlsx_df)
else:
    print("\nAll Banner Roster (XLSX) rows found a match.")

In [ ]:
# @title process unmatched rows
# @markdown Some users have a personal email address as their `login_id` in canvas resulting in a failure to match the rows between the two rosters. This code will attempt to match the unmatched rows via the `name` and `Student Name` fields from the canvas and banner files respectively.
import pandas as pd
import re

# Prepare an empty list to store newly merged rows
newly_merged_data = []

# Keep track of matched indices from the original unmatched DFs
newly_matched_csv_indices = set()
newly_matched_xlsx_indices = set()

# Function to clean names for better comparison
def clean_name(name):
    # Convert to string, lowercase, strip whitespace
    name = str(name).lower().strip()
    # Remove text in parentheses (like pronouns or preferred names)
    name = re.sub(r'\s*\([^\)]*\)', '', name)
    # Remove common suffixes/prefixes if necessary (e.g., M. or G. for middle names)
    name = re.sub(r'\s+[a-z]\.$', '', name) # Removes single letter followed by dot
    name = re.sub(r',', '', name) # Remove commas
    return name

print("\n--- Attempting additional matches using Name fields ---")

# Iterate through each unmatched CSV row
for idx_csv, row_csv in unmatched_csv_df.iterrows():
    csv_name_clean = clean_name(row_csv['name'])

    # Iterate through each unmatched XLSX row to find a match
    for idx_xlsx, row_xlsx in unmatched_xlsx_df.iterrows():
        xlsx_student_name_clean = clean_name(row_xlsx['Student Name'])

        # Check if the cleaned CSV name is a substring of the cleaned XLSX student name
        # and if the XLSX row hasn't been matched yet in this round
        if csv_name_clean in xlsx_student_name_clean and idx_xlsx not in newly_matched_xlsx_indices:
            # Combine the CSV row and the matched XLSX row
            combined_series = pd.concat([row_csv, row_xlsx], ignore_index=False)
            newly_merged_data.append(combined_series)

            # Mark both indices as matched
            newly_matched_csv_indices.add(idx_csv)
            newly_matched_xlsx_indices.add(idx_xlsx)
            break # Move to the next CSV row once a match is found

# Create DataFrame for newly found matches
newly_matched_df = pd.DataFrame(newly_merged_data)

if not newly_matched_df.empty:
    print("\nNewly matched rows using Name/Student Name:")
    display(newly_matched_df)

    # Update the main merged_df with these new matches
    global merged_df # Declare merged_df as global to modify it
    merged_df = pd.concat([merged_df, newly_matched_df], ignore_index=True)
    print(f"\nUpdated number of rows in combined DataFrame: {len(merged_df)}")
else:
    print("\nNo additional matches found using Name/Student Name.")

# Identify remaining unmatched rows
remaining_unmatched_csv_df = unmatched_csv_df[~unmatched_csv_df.index.isin(newly_matched_csv_indices)]
remaining_unmatched_xlsx_df = unmatched_xlsx_df[~unmatched_xlsx_df.index.isin(newly_matched_xlsx_indices)]

print("\n--- Remaining Unmatched Rows After Second Attempt ---")
if not remaining_unmatched_csv_df.empty:
    print("\nRemaining rows from Canvas Roster (CSV) without a match:")
    display(remaining_unmatched_csv_df)
else:
    print("\nAll Canvas Roster (CSV) rows are now matched.")

if not remaining_unmatched_xlsx_df.empty:
    print("\nRemaining rows from Banner Roster (XLSX) without a match:")
    display(remaining_unmatched_xlsx_df)
else:
    print("\nAll Banner Roster (XLSX) rows are now matched.")


In [ ]:
# @title Extract first and last name plus pronouns from the data
import re
import pandas as pd

def parse_student_name(student_name_str):
    if pd.isna(student_name_str):
        return None, None, None

    student_name_str = str(student_name_str).strip()

    first_name = None
    last_name = None
    pronoun = None

    # Try to extract pronoun first
    pronoun_match = re.search(r'Pronoun:\s*([^)]+)', student_name_str, re.IGNORECASE)
    if pronoun_match:
        pronoun = pronoun_match.group(1).strip()
        # Remove the pronoun part and anything related to 'Pref' if it's within the same parenthesis structure
        student_name_str_cleaned = re.sub(r'\s*\(.*?Pronoun:[^)]*\)', '', student_name_str, flags=re.IGNORECASE).strip()
    else:
        student_name_str_cleaned = student_name_str

    # Then extract last name and first name from the cleaned string
    name_parts_match = re.match(r'([^,]+),\s*(.+)', student_name_str_cleaned)
    if name_parts_match:
        last_name = name_parts_match.group(1).strip()
        first_name_full = name_parts_match.group(2).strip()

        # The first name might have middle initials or (Pref: ...) part, extract only the first name
        # Split by space or by the start of a parenthesis
        first_name_match = re.match(r'([^\s(]+)', first_name_full)
        if first_name_match:
            first_name = first_name_match.group(1).strip()
        else:
            first_name = first_name_full.split(' ')[0] # Fallback: take the first word

    return first_name, last_name, pronoun

# Apply the function to the 'Student Name' column
merged_df[['first_name', 'last_name', 'pronoun']] = merged_df['Student Name'].apply(lambda x: pd.Series(parse_student_name(x)))

print("Merged DataFrame with new 'first_name', 'last_name', and 'pronoun' columns:")
display(merged_df[['Student Name', 'first_name', 'last_name', 'pronoun']].head())

In [ ]:
display(merged_df)

In [ ]:
# merged_df = merged_df.rename({'section': 'ta_name'}, axis='columns')

# display(merged_df)

In [ ]:
# @title Update auth_code, ta_name, and section {"form-width":"20%"}
auth_code = "dEisTdS-PdV" # @param {"type":"string","placeholder":"Enter Auth_code for course"}
ta_name = "" # @param {"type":"string","placeholder":""}

pattern = r"PH-(.+)-(.+): Lab For Ph (.+)"
merged_df[['lab_course','section','lecture_course']] = merged_df['sections'].str.extract(pattern)

if 'auth_code' not in merged_df.columns:
  merged_df['auth_code'] = auth_code
if 'group_name' in merged_df.columns:
  merged_df.rename(columns={'group_name': 'ta_name'}, inplace=True)
  print("Column 'group_name' renamed to 'ta_name'.")

if ta_name != "":
  merged_df['ta_name'] = ta_name

output_df = merged_df[['first_name', 'last_name', 'pronoun', 'ID', 'Email', 'ta_name','section', 'auth_code']]
display(output_df)

# Export csv file for upload

In [ ]:
# @title Adjust dataframe to site formatting. {"form-width":"20%"}
export_csv = True # @param {"type":"boolean"}
folder_link = "https://drive.google.com/drive/folders/1AVkoBd38NIUnBk-sN7FCk72NT9iiLkK5?usp=drive_link" # @param {"type":"string","placeholder":"Enter the link to the google drive folder here"}
filename = "Online_Roster_Mar31-2026.csv" # @param {"type":"string","placeholder":"Enter filename here"}


folder_link_id = extract_file_id(folder_link)
print(f"folder_id {folder_link_id}")

if export_csv:
    # Initialize GoogleDocumentManager if not already done in this scope or if needed for specific operations
    if 'document_manager' not in locals() or document_manager is None:
        document_manager = GoogleDocumentManager()

    # Replace 'YOUR_GOOGLE_DRIVE_FOLDER_ID_HERE' with the actual Google Drive folder ID
    # where you want to save the CSV file.

    print(f"\nAttempting to upload '{filename}' to Google Drive...")
    file_id = document_manager.upload_dataframe_to_drive(output_df, filename, folder_link_id)
    if file_id:
        print(f"Successfully exported '{filename}' with File ID: {file_id}")
    else:
        print(f"Failed to export '{filename}'. Please check the folder ID and permissions.")

# Compare Rosters

In [ ]:
# @title report differences in csv files {"form-width":"20%"}
compare_rosters = False # @param {"type":"boolean"}

import pandas as pd

if compare_rosters:
  old_roster_link = "https://drive.google.com/file/d/16WGcd12URCYULRWtnaZ4SPrsFS579UAr/view?usp=drive_link" # @param {"type":"string","placeholder":"Enter link to old roster"}
  new_roster_link = "" # @param {"type":"string","placeholder":"Enter link to new roster"}
  # @markdown ---
  export_new = False # @param {"type":"boolean"}
  folder_link = "https://drive.google.com/drive/folders/1Kvgm95JK3k6YiFHV9TWyk_CHy5p5bWOG?usp=drive_link" # @param {"type":"string","placeholder":"Enter the link to the google drive folder here"}
  filename = "Roster_difference_Jan29-Feb9-2026.csv" # @param {"type":"string","placeholder":"Enter filename here"}

# @markdown ---
# @markdown Extract file IDs from the provided links, read the CSV files using the GoogleDocumentManager, and then compare them to identify rows present in the new roster but not in the old roster (new rows), and rows present in the old roster but not in the new roster (missing rows).

  old_roster_file_id = extract_file_id(old_roster_link)
  new_roster_file_id = extract_file_id(new_roster_link)

  print(f"Extracted Old Roster File ID: {old_roster_file_id}")
  print(f"Extracted New Roster File ID: {new_roster_file_id}")

  document_manager = GoogleDocumentManager()

  print("\nAttempting to read old roster CSV file...")
  df_old_roster = document_manager.read_file(old_roster_file_id, file_type='csv')
  if df_old_roster is not None:
      # print("Old Roster DataFrame head:")
      # display(df_old_roster.head())
      pass
  else:
      print("Failed to read old roster CSV file.")

  print("\nAttempting to read new roster CSV file...")
  df_new_roster = document_manager.read_file(new_roster_file_id, file_type='csv')
  if df_new_roster is None:
      print("Failed to read new roster CSV file.")
  else:
      # print("New Roster DataFrame head:")
      # display(df_new_roster.head())
      pass

  if df_new_roster is not None and df_old_roster is not None:

    # 1. Convert 'ID' column to string type in both dataframes
    df_old_roster['ID'] = df_old_roster['ID'].astype(str)
    df_new_roster['ID'] = df_new_roster['ID'].astype(str)

    # Get sets of IDs for comparison
    old_roster_ids = set(df_old_roster['ID'])
    new_roster_ids = set(df_new_roster['ID'])

    # 2. Identify new rows (present in new_roster but not in old_roster)
    new_ids = list(new_roster_ids - old_roster_ids)
    new_rows_df = df_new_roster[df_new_roster['ID'].isin(new_ids)].reset_index(drop=True)

    # 3. Identify missing rows (present in old_roster but not in new_roster)
    missing_ids = list(old_roster_ids - new_roster_ids)
    missing_rows_df = df_old_roster[df_old_roster['ID'].isin(missing_ids)].reset_index(drop=True)

    print("New Rows (in new_roster but not in old_roster):")
    if not new_rows_df.empty:
        display(new_rows_df)
    else:
        print("No new rows found.")

    print("\nMissing Rows (in old_roster but not in new_roster):")
    if not missing_rows_df.empty:
        display(missing_rows_df)
    else:
        print("No missing rows found.")

    if export_new:
      if not new_rows_df.empty:

        folder_link_id = extract_file_id(folder_link)
        print(f"folder_id {folder_link_id}")

        # Initialize GoogleDocumentManager if not already done in this scope or if needed for specific operations
        if 'document_manager' not in locals() or document_manager is None:
            document_manager = GoogleDocumentManager()

        print(f"\nAttempting to upload '{filename}' to Google Drive...")
        diff_file_id = document_manager.upload_dataframe_to_drive(new_rows_df, filename, folder_link_id)
        if diff_file_id:
            print(f"Successfully exported '{filename}' with File ID: {diff_file_id}")
        else:
            print(f"Failed to export '{filename}'. Please check the folder ID and permissions.")
      else:
        print("No new rows to export.")
  else:
    print("No changes to export.")
else:
  print('No comparison ran')